# AI-Codex: Spanish legal corpus build on Kaggle -> ready ChromaDB

One run fetches, chunks and vectorizes, and leaves the finished database in this version's **Output** tab:
1. **Fetch** (progress line per step): BOE consolidated legislation (`fetch_laws.py`), EU law in Spanish
   (`fetch_eu_law.py`) and, optionally, Tribunal Constitucional sentencias (`fetch_constitutional_court.py`).
2. **Check** the core codes are there (`check_corpus_coverage.py`).
3. **Chunk + vectorize** on the GPU -- `vectorize.py`, with the BM25 copy (`--export-lexical`).
4. **Package**: `legal_corpus_vectordb.zip` (the database) and `legal_corpus_jsonl.zip` (records, BOE cache, manifests).

**Before running** (panel on the right):
1. *Settings -> Accelerator*: **GPU T4 x2**. *Settings -> Internet*: **On**.
2. *Add-ons -> Secrets*: **GITHUB_TOKEN** (can read `zananiri/AI-IZ`) and **LEGAL_DATA_CONTACT_EMAIL** (goes in the
   fetchers' User-Agent; they refuse to run without it).
3. Set `CHROMADB_VERSION` below to the chromadb version on the computer you'll install on.
4. Time: the BOE's ~12,400 acts take ~3.5 h on a first run at one request per second, the TC's ~32,000 pages ~9 h --
   too much for one 12-hour session together. Run with `FETCH_TC = False` first; then attach this run's Output
   (*Add Input -> Your Work*) and run again with `FETCH_TC = True`: `REUSE_FETCH` restores the BOE cache and TC cache,
   so only what is new is fetched.
5. *Save Version -> Save & Run All (Commit)*. The run stops at the first failed step instead of packaging whatever is there.

**Install on your computer** when it finishes: `python scripts/legal_data/install_corpus.py <legal_corpus_vectordb.zip>`.


In [ ]:
BRANCH = "claude/ai-codex-spanish-market-htnmeb"   # or "main" once merged
CATEGORIES = "laws,regulations,eu_law"   # + regional_laws (autonomous communities), constitutional_court
BOE_CATEGORIES = "laws,regulations"      # what fetch_laws.py fetches (add regional_laws for ~3,600 more acts)
FETCH_TC = False                         # Tribunal Constitucional sentencias (~9 h on a first run)
CHROMADB_VERSION = "1.5.9"               # the chromadb version on the computer you'll install on
DEVICE = "cuda"                          # "cuda:0,cuda:1" uses both T4s
REUSE_FETCH = True                       # restore an attached earlier run's records and caches first


## 1. Code, packages and settings

In [ ]:
import os, pathlib, shutil, json, glob, subprocess
from kaggle_secrets import UserSecretsClient

if DEVICE.startswith("cuda"):
    import torch
    wanted = len(DEVICE.split(","))
    if torch.cuda.device_count() < wanted:
        raise RuntimeError(f"DEVICE={DEVICE!r} needs {wanted} GPU(s), found {torch.cuda.device_count()}: "
                           "Settings -> Accelerator -> GPU T4 x2, then run again")

def sh(command):
    """Runs a shell command with its output in this cell; raises (stopping the run) if it fails."""
    proc = subprocess.Popen(["bash", "-o", "pipefail", "-c", command], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait():
        raise RuntimeError(f"exit code {proc.returncode}: {command}")

secrets = UserSecretsClient()
token = secrets.get_secret("GITHUB_TOKEN")
os.environ["AICODEX_LEGAL_DATA_CONTACT_EMAIL"] = secrets.get_secret("LEGAL_DATA_CONTACT_EMAIL")
CLONE = "/kaggle/working/AI-IZ"
sh(f"git clone -q --depth 1 -b {BRANCH} https://{token}@github.com/zananiri/AI-IZ.git {CLONE}")
sh(f"git -C {CLONE} remote set-url origin https://github.com/zananiri/AI-IZ.git")
os.chdir(f"{CLONE}/AI-Codex")
sh("git log --oneline -1")
sh('pip install -q -e ".[legal,legal-data,dev]"')
sh(f'pip install -q "chromadb=={CHROMADB_VERSION}"')
sh("python -c \"import chromadb, sentence_transformers; print('chromadb', chromadb.__version__, '| sentence-transformers', sentence_transformers.__version__)\"")
VECTORDB = "/tmp/legal_corpus_vectordb"   # scratch space; only the zips go to /kaggle/working
pathlib.Path(VECTORDB).mkdir(parents=True, exist_ok=True)
DEVICE_ARG = f"--devices {DEVICE}" if "," in DEVICE else f"--device {DEVICE}"


## 2. Unit tests (informational -- a failure here doesn't stop the run)

In [ ]:
!python -m pytest tests/unit -q -p no:cacheprovider 2>&1 | tail -5


## 3. Restore an earlier run (records, BOE cache, TC cache), if attached

In [ ]:
if REUSE_FETCH:
    found = sorted(pathlib.Path("/kaggle/input").rglob("boe/cache"))
    if found:
        root = found[0].parent.parent
        for sub in ("boe", "constitutional_court", "laws", "regulations", "regional_laws", "eu_law", "_manifests"):
            if (root / sub).is_dir():
                shutil.copytree(root / sub, pathlib.Path("legal_txt") / sub, dirs_exist_ok=True)
        print("restored", root, "-- only new and updated acts will be fetched")
    else:
        print("REUSE_FETCH: nothing attached, fetching everything")


## 4. Fetch: BOE legislation, EU law, Tribunal Constitucional

In [ ]:
sh(f"python scripts/legal_data/fetch_laws.py --categories {BOE_CATEGORIES}")
sh("python scripts/legal_data/fetch_eu_law.py")
if FETCH_TC:
    sh("python scripts/legal_data/fetch_constitutional_court.py")
for path in sorted(glob.glob("legal_txt/_manifests/*_fetch_*.json")):
    run = json.load(open(path, encoding="utf-8"))
    print(f"{run['script']}: {run['status']} | counts {run['counts']}")
    for warning in run["warnings"][:5]:
        print("   warning:", warning)
sh("python scripts/legal_data/check_corpus_coverage.py")


## 5. Chunk and vectorize (resumable: re-running this cell continues where it stopped)

In [ ]:
categories = ",".join(c for c in CATEGORIES.split(",") if any(pathlib.Path("legal_txt", c).glob("*.jsonl")))
sh(f"python scripts/legal_data/vectorize.py --input legal_txt --vectordb {VECTORDB} --categories {categories} --dry-run")
sh(f"python scripts/legal_data/vectorize.py --input legal_txt --vectordb {VECTORDB} --categories {categories} {DEVICE_ARG}")


## 6. Test queries with metadata filters

In [ ]:
PROBES = [("laws", "¿Qué plazo de prescripción tiene la responsabilidad civil extracontractual?", '{"status": "in_force"}'),
          ("laws", "¿Cuál es la duración mínima de un contrato de alquiler de vivienda?", ""),
          ("regulations", "¿Qué requisitos tiene el registro de la jornada de trabajo?", ""),
          ("eu_law", "¿Cuándo es lícito el tratamiento de datos personales?", ""),
          ("constitutional_court", "derecho a la tutela judicial efectiva sin indefensión", "")]
for category, question, where in PROBES:
    if category in categories.split(","):
        where_arg = f"--where '{where}'" if where else ""
        sh(f'python scripts/legal_data/vectorize.py --vectordb {VECTORDB} --probe "{question}" --category {category} {where_arg} --top-k 3')


## 7. Output: the database and the records

In [ ]:
sh(f"python scripts/legal_data/vectorize.py --vectordb {VECTORDB} --categories {categories} --export-lexical")
info = json.load(open(f"{VECTORDB}/_build_info.json", encoding="utf-8"))
empty = [c for c in categories.split(",") if not info["categories"].get(c, {}).get("chunks")]
if empty:
    raise RuntimeError(f"no chunks for {empty}: not packaging an empty database")
shutil.make_archive("/kaggle/working/legal_corpus_vectordb", "zip", VECTORDB)
shutil.make_archive("/kaggle/working/legal_corpus_jsonl", "zip", "legal_txt")
print(json.dumps(info, ensure_ascii=False, indent=1))
sh("ls -lh /kaggle/working/*.zip")
print("On your computer: python scripts/legal_data/install_corpus.py <path to legal_corpus_vectordb.zip>")
